# Lab 8.2 Triangulation

In this lab you will capture two images using your camera (which you calibrated in Lab 3.1), estimate the relative pose between them, and triangulate points to create sparse "point cloud" 3D model.

## Instructions:

1. Using your calibrated camera, capture two images of some scene / object, making sure to translate the camera between the two images.

2. Match keypoints between the two images (e.g. using SIFT or ORB) and determine the essential matrix using `cv2.findEssentialMat`.

3. Use `cv2.recoverPose` to find the rotation and translation between the two images from the essential matrix.

4. Use `cv2.triangulatePoints` on the inlier feature matches to estimate 3D points.

5. Make an OBJ file (a text file with extension .obj) to store the 3D points.  Each line stores one point, with this format:
`v x y z`
where x,y,z are the 3D coordinates of the point.

You can view the OBJ file in an online viewer like [this one](https://3dviewer.net).

In [ ]:
import skimage
import numpy as np
import glob

from matplotlib import pyplot as plt
from imageio.v3 import imread
from skimage.feature import SIFT, match_descriptors

from cv2 import findEssentialMat, recoverPose, triangulatePoints, RANSAC

In [ ]:
# Get camera intrinsics matrix from Lab 3.1
INTRINSIC_MATRIX = np.array([[3.07051048e+03, 0.00000000e+00, 2.01159627e+03],
 [0.00000000e+00, 3.07051048e+03, 1.16114597e+03],
  [0.00000000e+00, 0.00000000e+00, 1.00000000e+00]])

In [ ]:
# Load the images
LEFT_PATH = 'left.jpg'
RIGHT_PATH = 'right.jpg'

# Load the images
left_img = skimage.color.rgb2gray(imread(LEFT_PATH))
right_img = skimage.color.rgb2gray(imread(RIGHT_PATH))

In [ ]:
# Get the keypoints and descriptors for both the left and right images
def get_key_and_desc(img, sift):
  sift.detect_and_extract(img)
  keypoints = sift.keypoints
  descriptors = sift.descriptors

  return keypoints, descriptors

SIFT_MODEL = SIFT()
left_keypoints, left_descriptors = get_key_and_desc(left_img, SIFT_MODEL)
right_keypoints, right_descriptors = get_key_and_desc(right_img, SIFT_MODEL)

In [ ]:
# Estimate the Essential Matrix
matches = match_descriptors(
    left_descriptors,
    right_descriptors,
    cross_check=True,
    max_ratio=0.8
)

# skimage keypoints are (row, col) = (y, x).
# OpenCV requires (x, y).
matched_left = left_keypoints[matches[:, 0]][:, ::-1].astype(np.float32)
matched_right = right_keypoints[matches[:, 1]][:, ::-1].astype(np.float32)

In [ ]:
essentialMatrix, mask = findEssentialMat(
    matched_left,
    matched_right,
    INTRINSIC_MATRIX,
    method=RANSAC,
    prob=0.999,
    threshold=1.0
)

In [ ]:
# Get the estimated rotation and translation vectors from this essential matrix
inlier_ct, pose_rotation, pose_translation, pose_mask = recoverPose(
    essentialMatrix,
    matched_left,
    matched_right,
    INTRINSIC_MATRIX,
    mask=mask
)

In [ ]:
# Estimate 3D points based on the inlier points
# Keep only pose inliers (the mask returned by recoverPose)
pose_inliers = pose_mask.ravel().astype(bool)

left_inliers = matched_left[pose_inliers]
right_inliers = matched_right[pose_inliers]

# Projection matrix for the left camera: K [I | 0]
P_left = INTRINSIC_MATRIX @ np.hstack((
    np.eye(3),
    np.zeros((3, 1))
))

# Projection matrix for the right camera: K [R | t]
P_right = INTRINSIC_MATRIX @ np.hstack((
    pose_rotation,
    pose_translation
))

# Returns homogeneous points with shape (4, N).
points_4d = triangulatePoints(
    P_left,
    P_right,
    left_inliers.T,
    right_inliers.T
)

# Convert [X, Y, Z, W] -> [X/W, Y/W, Z/W], shape (N, 3)
points_3d = (points_4d[:3] / points_4d[3]).T

In [ ]:
# Make an OBJ file (a text file with extension .obj)
# to store the 3D points. Each line stores one point,
# with this format: v x y z where x,y,z are the 3D coordinates of the point.

with open("reconstructed_points.obj", "w") as obj_file:
    for x, y, z in points_3d:
        obj_file.write(f"v {x:.6f} {y:.6f} {z:.6f}\n")